In [1]:
from iconclass import init

ic = init()

def walk(node):
    """Yield node and all descendants recursively."""
    yield node
    for child in node:
        yield from walk(child)

In [2]:
def tree_distance(node1, node2):
    """Distance = how many steps apart in the tree (parent/child)."""
    p1 = node1.path()
    p2 = node2.path()

    # convert to sets of notations
    n1 = [repr(n).split()[0] for n in p1]
    n2 = [repr(n).split()[0] for n in p2]

    # closest common ancestor depth
    common = set(n1) & set(n2)

    if not common:
        return 5  # unrelated → far distance

    # distance = steps to go up + steps to go down
    d1 = min(n1.index(c) for c in common)
    d2 = min(n2.index(c) for c in common)

    return d1 + d2


In [3]:
def text_match_score(label, notation, query_words):
    score = 0
    words = label.lower().split()

    for q in query_words:
        q = q.lower()

        # exact word
        if q in words:
            score += 50
            continue

        # substring match
        if q in label.lower():
            score += 20

        # match in notation
        if q in notation:
            score += 10

    return score

In [4]:
def prefix_bonus(base_notation, candidate_notation):
    # example: 31C21 vs 31C214 → strong bonus
    match_len = 0
    for a, b in zip(base_notation, candidate_notation):
        if a == b:
            match_len += 1
        else:
            break
    return match_len * 2  # small bonus per matching character

In [5]:
def search_iconclass_classic(query):
    query_words = query.lower().split()
    results = []

    # First, collect all nodes and their labels
    all_nodes = list(walk(ic))

    # Find strong matches first (nodes with highest text score)
    strong_matches = []
    for node in all_nodes:
        notation = repr(node).split()[0].lower()
        try:
            label = node("en").lower()
        except:
            label = ""

        tscore = text_match_score(label, notation, query_words)

        if tscore > 0:
            strong_matches.append((tscore, node, notation, label))

    # Score everything against strong matches (hierarchy-based)
    final_results = []
    for tscore, node, notation, label in strong_matches:

        # find best tree distance to any strong match
        best_tree_bonus = 0
        for _, strong_node, strong_notation, _ in strong_matches:
            d = tree_distance(node, strong_node)
            # closer = bigger bonus
            best_tree_bonus = max(best_tree_bonus, max(20 - d*5, 0))

        # prefix bonus
        best_prefix = 0
        for _, strong_node, strong_notation, _ in strong_matches:
            best_prefix = max(best_prefix, prefix_bonus(strong_notation, notation))

        final_score = tscore + best_tree_bonus + best_prefix
        final_results.append((final_score, node))

    # sort
    final_results.sort(key=lambda x: x[0], reverse=True)
    return final_results

In [ ]:
query = input("Enter your Iconclass search query: ")

results = search_iconclass_classic(query)

for score, node in results[:20]:
    notation = repr(node).split()[0]
    try:
        label = node("en")
    except:
        label = "(no label)"
    print(f"{score:3d}  {notation} → {label}")